# Logistic Regression

# Why Not Linear Regression for Classification?

- a regression method cannot return a meaningful output, since the model output can be [$-\infty$, $+\infty$]

# Why Logistic Regression?

- easy interpretation
- cheap to train the model
- less likely to overfit to data

# How to Convert Linear to Logistic Regression

- Let's start with the original linear regression formula
  - $\hat{f}(x) = w_0 + w^T*x$
- We must ensure the range of this function is between [0, 1]
  - We can pass our linear function into a sigmoid function to ensure this
  - $p(x) = \frac{e^{w_0 + w^T*x}}{1 + e^{w_0 + w^T*x}} = \frac{1}{1 + e^{-(w_0 + w^T*x)}}$
- Below is how the output from the linear regression changes when we use the original (left) vs. this sigmoid version (right):
  - <img src="images/plot-4.png" width="600"/>

# Probability, Odds, Log-Odds, Likelihood

- First and foremost, these are different ways to quantify the likelihood of an event occurring
- Our logistic regression model's output is a **probability** (i.e., the chance of an event occuring)
  - $p(x) = \frac{e^{w_0 + w^T*x}}{1 + e^{w_0 + w^T*x}}$
- Above formula, given manipulation, is equivalent to....
  - Odds: $\frac{p}{1 - p} = e^{w_0 + w^T*x}$, which is telling me how likely the event is to happen relative to it not happening. Example: 0.25 (1/4), means that the event is 4x likely not to occur than occur.
  - Log-Odds: $\log(\frac{p}{1 - p}) = w_0 + w^T*x$
- Difference Between Probability and Likelihood
  - <img src="images/plot-5.png" width="600"/>

# Maximum Likelihood Estimator (MLE) for Logistic Regression

- $P(y = 1 | w,x) = \frac{1}{1 + e^{-(w_0 + w^T*x)}}$
- $P(y = -1 | w,x) = \frac{1}{1 + e^{(w_0 + w^T*x)}}$, switching y = 0 to y = -1 for easier derivative, because of this property of the sigmoid function
- $P(y = y | w,x) = \frac{1}{1 + e^{-y(w_0 + w^T*x)}}$, this is equivalent to the first two bullet points
- $L(w_0, w^T|x_i, y_i) = \prod_{i=1}^{n}\frac{1}{1 + e^{-y(w_0 + w^T*x)}}$, the likelihood function for all data points, we want to maximize
- $L(w_0, w^T|x_i, y_i) = \sum_{i=1}^{n}\log(1) - \log(1 + e^{-y(w_0 + w^T*x)})$, converting to log for easier optimization
- $L(w_0, w^T|x_i, y_i) = \sum_{i=1}^{n}-\log(1 + e^{-y(w_0 + w^T*x)})$, cancelling out
- $L(w_0, w^T|x_i, y_i) = \sum_{i=1}^{n}\log(1 + e^{-y(w_0 + w^T*x)})$, multiplying by negative, so now we are finding the same optimal values for the weights to minimize the likelihood function instead of maximizing it
- there is no closed form solution, so we'll need to use gradient or stochastic gradient descent to find optimal weight values

# Evaluation Metrics

## Precision
- Explanation
  - Measures how reliable are your positive predictions. Out of all the observations that we predict positive, how many of them are actually positive.
- Formula
  - $\mathrm{Precision} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}$
  - $TP$ = observations that are both labeled and predicted to be positive
  - $FP$ = observations that are labeled negative, but predicted to be positive
- Advantages and Disadvantages
  - one of the most important metrics in classification, especially my industry
## Recall
- Explanation
  - Measures how well the model does at capturing our positive population.
- Formula
  - $\mathrm{Recall} = \frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN}}$
  - $TP$ = observations that are both labeled and predicted to be positive
  - $FN$ = observations that are labeled positive, but predicted to be negative
- Advantages and Disadvantages
  - Well if the data is imbalanced, then we may be great at capturing the positive group (high recall), but the precision could be really bad because we have a lot of labelled-negatives that we predict positive.
## AUC
- Explanation
  - The two above metrics are used when we make a rule that any predicted probability greater than a threshold is predicted positive, otherwise predicted negative.
  - So, its useful to understand how our model behaves at multiple thresholds, which is where AUC (Area Under the Curve) comes into play.
  - This tells us the overall recall and spec over all thresholds.
- Formula
- Advantages and Disadvantages
## AUPRC
- Explanation
  - Same as above, but its recall and precision. And this tells us more accurately our precision for making predictions.
- Formula
- Advantages and Disadvantages
## Confusion Matrix
- Explanation
  - evaluating classification models. It's a table that shows how well your model's predictions match the actual outcomes.
- Formula
  - $\begin{array}{c|cc}
& y = 1 & y = 0 \\
\hline
\hat{y} = 1 & TP & FP \\
\hat{y} = 0 & FN & TN \\
\end{array}$
  - TP (True Positive): how many were predicted and labeled positive
  - FP (False Positive): how many were predicted positive, but labeled negative
  - FN (False Negative): how many were predicted negative, but labeled positive
  - TN (True Negative): how many were predicted and labeled negative

# Practice

In [8]:
import numpy as np
import pandas as pd
import polars as pl
import pyarrow
from ISLP import load_data
import matplotlib.pyplot as plt
import matplotlib 

In [9]:
# load dataset
Weekly = load_data("Weekly")
Weekly = pl.from_pandas(Weekly)
Weekly.head()

Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
i64,f64,f64,f64,f64,f64,f64,f64,cat
1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.27,"""Down"""
1990,-0.27,0.816,1.572,-3.936,-0.229,0.148574,-2.576,"""Down"""
1990,-2.576,-0.27,0.816,1.572,-3.936,0.1598375,3.514,"""Up"""
1990,3.514,-2.576,-0.27,0.816,1.572,0.16163,0.712,"""Up"""
1990,0.712,3.514,-2.576,-0.27,0.816,0.153728,1.178,"""Up"""


Produce some numerical and graphical summaries of the Weekly data. Do there appear to be any patterns?

In [3]:
len(Weekly.select(pl.col("Year")).unique()) # there are 21 years

21